[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-10-prefect-cloud-server.ipynb#scrollTo=a1b2c3d4)

---
# Day 10 · Prefect Cloud vs Self-Hosted Server — Setup and Tradeoffs
**certified-journeys / prefect-certified** · Review

> **Goal for today:** Understand the architectural differences between Prefect Cloud and a self-hosted server, connect a local environment to Prefect Cloud using an API key, and articulate the tradeoffs that govern which option to choose.


In [ ]:
%pip install -q "prefect>=2.14" prefect-shell requests


---
## Step 1 · Architecture overview — Cloud vs self-hosted

Prefect has two hosting modes for its **orchestration layer** (the API server + database that stores runs, deployments, and logs).

| Dimension | Prefect Cloud | Self-Hosted Server |
|---|---|---|
| Who runs the server | Prefect (SaaS) | You — on any VM or K8s cluster |
| Setup time | 5 minutes (sign up + API key) | 30 min+ (Docker / K8s / DB provisioning) |
| Authentication | Workspaces + API keys + RBAC | DIY (reverse proxy / OAuth) |
| Audit logs | ✓ built-in | ✗ manual log aggregation |
| SLA monitoring | ✓ automations + alerts | ✗ manual setup |
| Cost | Free tier + paid plans | Infrastructure cost only |
| Data residency | Prefect's cloud (US/EU) | Your own infrastructure |
| Upgrades | Automatic | Manual (`prefect server start` new version) |

**Key insight:** The *workers* always run in your infrastructure regardless of hosting mode — only the orchestration layer (API + DB + UI) moves to Prefect's servers when using Cloud.


In [ ]:
# Inspect current Prefect configuration — shows which server/cloud your client targets
import subprocess
import os

# Print current Prefect settings that relate to server configuration
settings_to_check = [
    "PREFECT_API_URL",
    "PREFECT_CLOUD_API",
    "PREFECT_API_KEY",           # will be empty until you log in
    "PREFECT_SERVER_ANALYTICS_ENABLED",
]

print("Current Prefect environment settings:")
print("-" * 48)
for key in settings_to_check:
    val = os.environ.get(key, "<not set>")
    # Mask API key — never log credentials in plain text
    if "KEY" in key and val != "<not set>":
        val = val[:8] + "..." + val[-4:] if len(val) > 12 else "***"
    print(f"  {key:<40} = {val}")

print()
# Show the active Prefect profile
result = subprocess.run(
    ["python", "-m", "prefect", "profile", "ls"],
    capture_output=True, text=True
)
print("Active Prefect profiles:")
print(result.stdout or result.stderr)


**What just happened?**
- `PREFECT_API_URL` determines which server your Prefect client talks to — empty means ephemeral/local
- For Prefect Cloud it will look like `https://api.prefect.cloud/api/accounts/<id>/workspaces/<id>`
- **API key** is always masked — never log or commit credentials; Prefect stores them encrypted in `~/.prefect/profiles.toml`
- `prefect profile ls` shows named configuration profiles — you can have one for local dev and one for Cloud


---
## Step 2 · Connecting to Prefect Cloud — the login flow

Connecting your local environment to Prefect Cloud is a three-step process:

```
1. Create account → app.prefect.cloud → sign up (free)
2. Generate API key → Settings → API Keys → + Add API Key
3. prefect cloud login --key <your-key>
```

Under the hood, `prefect cloud login` does:
1. Validates the key against `https://api.prefect.cloud/api/me/`
2. Writes the key + workspace URL to `~/.prefect/profiles.toml`
3. Sets `PREFECT_API_URL` in the active profile

After login every `prefect` CLI call and Python SDK call goes to your Cloud workspace automatically — you don't touch the URL directly.

**API key scopes (fine-grained tokens)**

| Scope | What it allows |
|---|---|
| `read` | Read flows, runs, deployments — good for dashboards |
| `write` | Create/update deployments, trigger runs |
| `manage` | Full admin — create workspaces, invite members |

Best practice: CI/CD pipelines get a `write` key. Monitoring dashboards get a `read` key.


In [ ]:
# Simulate what prefect cloud login does — inspect the profiles.toml structure
# (We use a local ephemeral server here so no real Cloud API key is needed)

import os
import pathlib
import json

profiles_path = pathlib.Path.home() / ".prefect" / "profiles.toml"

print("Prefect profiles file location:")
print(f"  {profiles_path}")
print(f"  Exists: {profiles_path.exists()}")

if profiles_path.exists():
    content = profiles_path.read_text()
    # Mask any API keys in the output
    import re
    masked = re.sub(r'(PREFECT_API_KEY\s*=\s*")[^"]+"', r'\1***"', content)
    print("\nprofiles.toml content (keys masked):")
    print(masked[:1200])  # first 1200 chars
else:
    print("\nNo profiles.toml yet — it's created on first 'prefect cloud login'")
    print("\nExpected structure after 'prefect cloud login --key pnu_xxx':")
    example = """
active = "cloud"

[profiles.default]
PREFECT_API_URL = ""

[profiles.cloud]
PREFECT_API_URL = "https://api.prefect.cloud/api/accounts/<account-id>/workspaces/<workspace-id>"
PREFECT_API_KEY = "pnu_..."
"""
    print(example)

# Show how to switch profiles programmatically
print("\nCLI profile commands:")
commands = [
    ("prefect cloud login --key <api-key>",  "Authenticate and set cloud profile"),
    ("prefect cloud logout",                  "Remove cloud credentials"),
    ("prefect profile ls",                    "List all profiles"),
    ("prefect profile use cloud",             "Activate 'cloud' profile"),
    ("prefect profile use default",           "Switch back to local server"),
    ("prefect cloud workspace ls",            "List your Cloud workspaces"),
    ("prefect cloud workspace set",           "Change active workspace"),
]
for cmd, desc in commands:
    print(f"  {cmd:<52} # {desc}")


**What just happened?**
- `~/.prefect/profiles.toml` stores named profiles — each profile can point to a different server
- **`active`** key at the top selects the current profile — switching is instant with `prefect profile use`
- The Cloud workspace URL embeds your account ID and workspace ID — this is what scopes your data
- **Never commit** `profiles.toml` to git — it contains your API key in plain text


---
## Step 3 · Re-running an existing flow and verifying it in the Cloud UI

Once logged in, any flow run you trigger locally appears in the Cloud UI automatically — no code changes required. The Prefect client intercepts flow execution and sends metadata (not your data) to the Cloud API.

**What metadata is sent to the Cloud API:**

| Sent to Cloud | NOT sent to Cloud |
|---|---|
| Flow run name, state, timestamps | Your data / DataFrame contents |
| Task states and durations | Secrets / environment variables |
| Log messages (`logger.info(...)`) | Source code |
| Parameters (their values) | Return values (only in result storage) |

This is a critical distinction for compliance: your **business data never leaves your infrastructure**.


In [ ]:
# Run a flow locally with the ephemeral server (Colab-safe)
# In a Cloud-connected environment this would appear in app.prefect.cloud

from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
from datetime import datetime
import json

@task(retries=1, retry_delay_seconds=2)
def extract_records(source: str, limit: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"Extracting {limit} records from '{source}'")
    # Simulated records — replace with real DB/API call in production
    return [
        {"id": i, "source": source, "value": round(i * 2.718, 3)}
        for i in range(1, limit + 1)
    ]

@task
def transform(records: list[dict], multiplier: float) -> dict:
    total = sum(r["value"] * multiplier for r in records)
    return {"count": len(records), "total": round(total, 4), "multiplier": multiplier}

@flow(name="cloud-demo-pipeline", log_prints=True)
def cloud_demo_pipeline(
    source: str = "api-v1",
    limit: int = 20,
    multiplier: float = 1.5,
) -> dict:
    """Demo pipeline — re-run this after connecting to Cloud to see it in the UI."""
    started = datetime.utcnow().isoformat()
    records = extract_records(source=source, limit=limit)
    summary = transform(records=records, multiplier=multiplier)
    print(f"Pipeline complete at {started}: {summary}")
    return summary

# Run with ephemeral server (no Cloud connection required for Colab)
with prefect_test_harness():
    result = cloud_demo_pipeline(source="colab-test", limit=10, multiplier=2.0)
    print(f"\nReturn value: {json.dumps(result, indent=2)}")
    print("\nAfter 'prefect cloud login', run this same cell — the run will appear in app.prefect.cloud")


**What just happened?**
- `prefect_test_harness()` substitutes the real Cloud API with an in-process server — identical code paths
- When connected to Cloud, the flow run record (name, state, logs, parameters) appears in `app.prefect.cloud` under your workspace
- **Your data** (the records list, the return value) never leaves your machine — only metadata is sent
- The `log_prints=True` flag means every `print()` in the flow is captured as a structured log entry in the UI


---
## Step 4 · Five Cloud vs self-hosted feature differences

These are the five features that most often drive the hosting decision:

| # | Feature | Prefect Cloud | Self-Hosted |
|---|---|---|---|
| 1 | **RBAC** | Workspace roles: Owner / Editor / Viewer. Per-workspace team scoping | None — single-tenant, no access control |
| 2 | **Audit logs** | All API calls logged with actor + timestamp; exportable | Must aggregate server logs manually (ELK stack etc.) |
| 3 | **SLA monitoring** | Automations trigger alerts when runs are late or crash | Build your own alerting against the server DB |
| 4 | **Multi-region / HA** | Globally distributed, 99.9% SLA, zero ops | You provision HA (replicas, load balancers, failover) |
| 5 | **Managed upgrades** | Cloud always runs latest stable — no action needed | Pin and upgrade yourself; test migrations each time |

**When to choose self-hosted:**
- Strict data residency requirements (financial, healthcare)
- Air-gapped / VPC-only environments
- Very high flow-run volume where Cloud costs exceed infra costs

**When to choose Cloud:**
- Teams that want zero infrastructure to maintain
- Projects that need RBAC from day 1
- Startups / small teams where ops burden matters


In [ ]:
# Programmatically inspect the Cloud API capabilities (mocked for Colab)
# In a real Cloud workspace these calls would return live data

import json
from datetime import datetime, timezone

# Simulate the workspace summary you'd get from the Cloud API
# Real call: GET https://api.prefect.cloud/api/accounts/{id}/workspaces/{id}/
mock_workspace_summary = {
    "workspace": {
        "name": "my-team-workspace",
        "handle": "my-team",
        "created": "2024-01-15T08:00:00Z",
        "plan": "free",
        "limits": {
            "flow_runs_per_month": 500,        # free tier limit
            "retention_days": 7,               # log/run history retention
            "workspaces": 1,                   # one workspace on free plan
        },
    },
    "rbac_roles": ["owner", "editor", "viewer"],
    "audit_log_enabled": False,               # audit logs = paid feature
    "automations_enabled": True,              # available on free tier
    "sla_monitoring_enabled": False,          # paid feature
}

# Simulate workspace feature comparison
plans = {
    "Free": {
        "price_per_month": 0,
        "flow_runs_per_month": 500,
        "workspaces": 1,
        "rbac": True,
        "audit_logs": False,
        "sla_monitoring": False,
        "retention_days": 7,
    },
    "Pro": {
        "price_per_month": 19,
        "flow_runs_per_month": 2000,
        "workspaces": 3,
        "rbac": True,
        "audit_logs": True,
        "sla_monitoring": True,
        "retention_days": 90,
    },
    "Self-Hosted": {
        "price_per_month": "infra_only",
        "flow_runs_per_month": "unlimited",
        "workspaces": "unlimited",
        "rbac": False,
        "audit_logs": False,
        "sla_monitoring": False,
        "retention_days": "unlimited",
    },
}

print("Prefect plan comparison:")
print(f"{'Feature':<28} {'Free':>12} {'Pro':>12} {'Self-Hosted':>14}")
print("-" * 68)
feature_labels = [
    ("price_per_month",       "Price ($/mo)"),
    ("flow_runs_per_month",   "Runs/month"),
    ("workspaces",            "Workspaces"),
    ("rbac",                  "RBAC"),
    ("audit_logs",            "Audit logs"),
    ("sla_monitoring",        "SLA monitoring"),
    ("retention_days",        "Run retention (days)"),
]
for key, label in feature_labels:
    vals = [str(plans[p][key]) for p in ["Free", "Pro", "Self-Hosted"]]
    print(f"  {label:<26} {vals[0]:>12} {vals[1]:>12} {vals[2]:>14}")

print()
print("Active workspace summary (mock):")
print(json.dumps(mock_workspace_summary, indent=2))


**What just happened?**
- The plan comparison makes the tradeoffs concrete: Free tier = 500 runs/month, 7-day retention — plenty for personal projects
- **Audit logs** and **SLA monitoring** are paid features — critical for enterprise compliance but not needed in development
- Self-hosted has no built-in RBAC — you'd need a reverse proxy + auth layer to restrict access
- The 500 free-tier runs/month is per workspace — a new workspace resets the counter


---
## Step 5 · Workspace isolation — multi-team setup

A **workspace** is a logical boundary in Prefect Cloud:

- Each workspace has its own flows, deployments, runs, blocks, and work pools
- Members can belong to multiple workspaces with different roles in each
- API keys are scoped to a workspace — a key from workspace A cannot access workspace B

**Common multi-team patterns:**

| Pattern | How workspaces map |
|---|---|
| Environment isolation | One workspace per env: `dev`, `staging`, `prod` |
| Team isolation | One workspace per team: `data-engineering`, `ml-platform` |
| Hybrid | One `prod` workspace shared, per-team `dev` workspaces |

Environment isolation is the most common pattern — it prevents dev runs from polluting prod metrics and lets you give engineers full access to dev without prod access.


In [ ]:
# Model workspace isolation — show how profiles map to workspaces

import textwrap

# This is what ~/.prefect/profiles.toml looks like after connecting
# to two workspaces (one per environment)
multi_workspace_profiles = textwrap.dedent("""
    # ~/.prefect/profiles.toml
    active = "cloud-prod"   # currently pointing at production workspace

    [profiles.default]
    # Empty = local ephemeral server (for unit tests, quick experiments)
    PREFECT_API_URL = ""

    [profiles.cloud-dev]
    # Development workspace — engineers can freely experiment here
    PREFECT_API_URL   = "https://api.prefect.cloud/api/accounts/acct-xxx/workspaces/ws-dev-yyy"
    PREFECT_API_KEY   = "pnu_dev_key_..."

    [profiles.cloud-prod]
    # Production workspace — restricted to CI/CD and on-call engineers
    PREFECT_API_URL   = "https://api.prefect.cloud/api/accounts/acct-xxx/workspaces/ws-prod-zzz"
    PREFECT_API_KEY   = "pnu_prod_key_..."
""").strip()

print("Multi-workspace profiles.toml example:")
print(multi_workspace_profiles)

print()
print("Workspace switching commands:")
workspace_cmds = [
    ("prefect profile use cloud-dev",    "Target dev workspace"),
    ("prefect profile use cloud-prod",   "Target prod workspace"),
    ("prefect cloud workspace ls",       "List workspaces in your account"),
    ("prefect cloud workspace set",      "Interactive workspace selector"),
]
for cmd, desc in workspace_cmds:
    print(f"  {cmd:<44} # {desc}")

# Show workspace-level RBAC mapping
print()
print("Workspace RBAC roles:")
roles = [
    ("Owner",  "Full access — create/delete workspaces, manage billing, invite members"),
    ("Editor", "Create/update flows, deployments, blocks — cannot manage team members"),
    ("Viewer", "Read-only — can see runs and logs but cannot trigger or modify anything"),
]
for role, desc in roles:
    print(f"  {role:<10} {desc}")

# Demonstrate how environment isolation prevents cross-workspace contamination
print()
print("Isolation guarantee:")
print("  A flow run created in 'cloud-dev' workspace NEVER appears in 'cloud-prod'.")
print("  API keys are workspace-scoped — prod keys cannot read dev workspace data.")
print("  Work pools, blocks, and automations are fully workspace-local.")


**What just happened?**
- **Profiles** are the local mechanism for switching between workspaces — one `prefect profile use` command and all CLI/SDK calls go to the new target
- **RBAC roles** are per-workspace: you can be an Owner in dev but only a Viewer in prod
- Work pools, blocks, and automations are isolated per workspace — a `slack-webhook` block in dev is not visible in prod
- CI/CD systems should store the workspace API key as a secret env var and never use a personal key


---
## Step 6 · Self-hosted server setup — the key steps

When you choose self-hosted, you run the Prefect server as a long-lived process backed by a PostgreSQL database.

**Minimum production self-hosted setup:**

```bash
# 1. Start the server (default: SQLite for dev, PostgreSQL for prod)
prefect server start --host 0.0.0.0 --port 4200

# 2. Configure all clients to point to your server
export PREFECT_API_URL="http://your-server:4200/api"

# 3. (Optional) Use Docker Compose for PostgreSQL + server
# docker-compose up -d
```

**Production considerations:**

| Concern | Self-hosted solution |
|---|---|
| Database | PostgreSQL 14+ (not SQLite) — configure `PREFECT_API_DATABASE_CONNECTION_URL` |
| Persistence | Mount a volume — SQLite default is in-container and lost on restart |
| TLS | Terminate at nginx/ALB — Prefect server speaks HTTP internally |
| Auth | Reverse proxy with OAuth/OIDC — server has no built-in auth |
| HA | Run 2+ server replicas behind a load balancer sharing one PostgreSQL DB |


In [ ]:
# Demonstrate self-hosted server via the ephemeral in-process mode
# Production equivalent: prefect server start --host 0.0.0.0 --port 4200

from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
import asyncio

@task
def ingest(table: str, rows: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"Ingesting {rows} rows into '{table}'")
    return [{"table": table, "row_id": i, "val": i * 42} for i in range(rows)]

@task
def validate_rows(rows: list[dict]) -> int:
    """Return number of rows that pass validation (val > 0)."""
    return sum(1 for r in rows if r["val"] > 0)

@flow(name="self-hosted-demo", log_prints=True)
def self_hosted_demo(
    table: str = "events",
    rows: int = 50,
) -> dict:
    """Flow that would run identically on Cloud or self-hosted — only the API URL changes."""
    raw = ingest(table=table, rows=rows)
    valid_count = validate_rows(rows=raw)
    result = {"table": table, "total": rows, "valid": valid_count}
    print(f"Ingestion complete: {result}")
    return result

# Run against the in-process server
# Production: PREFECT_API_URL=http://your-server:4200/api python this_script.py
with prefect_test_harness():
    r1 = self_hosted_demo(table="orders",  rows=30)
    r2 = self_hosted_demo(table="returns", rows=15)
    print(f"\nRun 1: {r1}")
    print(f"Run 2: {r2}")
    print(f"\nTotal valid rows ingested: {r1['valid'] + r2['valid']}")
    print("\nOn a real self-hosted server these runs would be visible at http://localhost:4200")


**What just happened?**
- The flow code is **identical** whether using Cloud or self-hosted — the only difference is `PREFECT_API_URL`
- `prefect_test_harness()` replicates the self-hosted in-process server — same code paths, no network
- In production, replace the context manager with `os.environ["PREFECT_API_URL"] = "http://your-server:4200/api"` and remove the `with` block
- **Portability** is Prefect's core promise: write once, run anywhere — local, self-hosted, or Cloud


---
## Challenge

You are designing the hosting setup for a new data pipeline project with the following constraints:
- Team of 4 engineers (2 junior, 2 senior)
- Budget: keep infrastructure costs under $50/month
- Requirement: engineers should not be able to delete production deployments
- Requirement: audit log of who triggered which run is needed for compliance

**Your tasks:**
1. Write a Python dict `hosting_decision` that captures which option (Cloud vs self-hosted) you'd choose and lists 3 reasons
2. Write a second dict `workspace_plan` that describes your workspace structure (how many workspaces, what each is called, which RBAC role each engineer type gets)
3. Write a function `profile_config(workspace_name: str, api_url: str, api_key: str) -> str` that generates the `profiles.toml` snippet (as a string) for a single workspace profile
4. Run the function for two workspaces: `"cloud-dev"` and `"cloud-prod"` with placeholder URLs and keys, and print the result


In [ ]:
# Challenge: your solution here

# Step 1: hosting decision
hosting_decision = {
    "choice": None,  # "cloud" or "self-hosted"
    "reasons": [
        # YOUR REASONS HERE
    ],
}

# Step 2: workspace plan
workspace_plan = {
    # YOUR WORKSPACE PLAN HERE
}

# Step 3 & 4: profile config generator
def profile_config(workspace_name: str, api_url: str, api_key: str) -> str:
    # YOUR CODE HERE
    pass

# Test the function
# YOUR CODE HERE


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Cloud vs self-hosted | Same Prefect code; only `PREFECT_API_URL` changes |
| API key login | `prefect cloud login --key <key>` writes to `~/.prefect/profiles.toml` |
| Profiles | Named configs — switch with `prefect profile use <name>` |
| Workspace isolation | Separate flows, runs, blocks per workspace — dev/prod isolation |
| RBAC | Owner / Editor / Viewer scoped per workspace |
| Audit logs | Cloud paid feature — logs all API actions with actor + timestamp |
| SLA monitoring | Cloud automations alert when runs are late or fail |
| Data privacy | Only metadata (run state, logs, params) sent to Cloud — never business data |

> **Tip:** Prefect Cloud's managed server means zero infrastructure to maintain — for personal projects and small teams, the free tier covers everything you need.

---
## What's next
**Day 11** → Notifications: set up Slack and email alerts when flows fail or succeed, and implement `on_failure` / `on_completion` hooks for in-process callbacks.

Mark Day 10 complete in your [tracker](../index.html).
